## Task 1: Loading the data, pre-processing and augmentation

import all dependencies, PyTorch's `ImageFolder` and `transforms` are used for convenient dataset loading 
and preprocessing

In [ ]:
import torch
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

We load the training, validation, and test sets from the KneeOA dataset using PyTorch's 
`ImageFolder`. Since the images are grayscale X-ray images, we convert them to a single 
channel using `transforms.Grayscale()`, as the three RGB channels are identical.

In [ ]:
base_path = "KneeOA/data_project"

gray_transform = transforms.Compose(
    [transforms.Grayscale(num_output_channels=1), transforms.ToTensor()]
)

train_dataset = ImageFolder(root=base_path + "/train", transform=gray_transform)
val_dataset = ImageFolder(root=base_path + "/val", transform=gray_transform)
test_dataset = ImageFolder(root=base_path + "/test", transform=gray_transform)
test_no_normalization_dataset = ImageFolder(root=base_path + "/test", transform=gray_transform) #for displaying images without normalization after baseline model

Next we take a look at the size and class distribution of each split.

In [ ]:
# basic info


def get_counts(dataset):
    arr = np.array([label for _, label in dataset])
    values, count = np.unique(arr, return_counts=True)
    counts = dict(zip(values.tolist(), count.tolist()))
    return counts


print("Train size:", len(train_dataset))
print("Train distribution:", get_counts(train_dataset), "\n")
print("Test size:", len(test_dataset))
print("Test distribution:", get_counts(test_dataset), "\n")
print("Val size:", len(val_dataset))
print("Val distribution:", get_counts(val_dataset), "\n")

We visualize a few samples from each class to get a feel for the data. 
We also verify that the grayscale conversion and tensor normalization 
were applied correctly: the shape should be `[1, H, W]` and pixel values should be in teh interval of `[0, 1]`.

In [ ]:



def show_samples(dataset, n=5):

    all_labels = np.array([label for _, label in dataset])
    idx_0 = np.where(all_labels == 0)[0][:n]
    idx_1 = np.where(all_labels == 1)[0][:n]

    selected_indices = np.concatenate([idx_0, idx_1])
    fig, axes = plt.subplots(2, n, figsize=(15, 6))
    axes_flat = axes.flatten()
    for plot_idx, dataset_idx in enumerate(selected_indices):
        img, label = dataset[dataset_idx]

        ax = axes_flat[plot_idx]
        ax.imshow(img.squeeze(), cmap="gray")
        ax.set_title(f"Label: {label}")
        ax.axis("off")

    plt.show()


show_samples(train_dataset, 5)
raw_img, _ = train_dataset[0]
print(f"Original shape: {raw_img.shape}, value range: {raw_img.min():.3f} -> {raw_img.max():.3f}")

We compute the global mean and standard deviation of the pixel intensities over the training set. 
These values will later be used for normalization.

In [ ]:
def compute_stats(dataset):
    loader = DataLoader(dataset, batch_size=64, shuffle=False)

    mean = 0.0
    std = 0.0
    total_images = 0

    for images, _ in loader:
        batch_size = images.size(0)
        images = images.view(batch_size, -1)

        mean += images.mean(1).sum()
        std += images.std(1).sum()
        total_images += batch_size

    mean /= total_images
    std /= total_images

    return mean.item(), std.item()


mean, std = compute_stats(train_dataset)
print("Intensity statistics:\n")
print(f"Global mean: {mean:.3f}")
print(f"Global std: {std:.3f}")

We define the preprocessing pipeline: images are resized to 128×128 pixels, converted to 
grayscale, and normalized using the mean and standard deviation computed from the training set.
We also define a version without normalization for visualization purposes, since normalized 
pixel values are not meaningful to display.

In [ ]:
preprocessing = transforms.Compose(
    [
        transforms.Resize((128, 128)),
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[mean], std=[std]),
    ]
)

preprocessing_no_normalization = transforms.Compose(
    [
        transforms.Resize((128, 128)),
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
    ]
)
train_dataset.transform = preprocessing_no_normalization
show_samples(train_dataset, n=5)
train_dataset.transform=preprocessing
transformed_img, _ = train_dataset[0]
print(
    f"preprocessed shape: {transformed_img.shape},value range: {transformed_img.min():.3f} -> {transformed_img.max():.3f}"
)

We implement two types of data augmentation to increase the diversity of the training set 
and reduce overfitting:

- **Geometric transforms**: random rotation, horizontal flip, and affine transformations 
(translation and scaling) to simulate variability in patient positioning.
- **Intensity transforms**: random brightness and contrast adjustments to simulate 
variability in X-ray exposure settings.

Both are applied randomly with a probability of 0.7. Augmentation is only applied to the 
training set, not to the validation or test sets, as we want to evaluate on unmodified images.

In [ ]:
geometric_transforms = transforms.RandomApply(
    [
        transforms.RandomRotation(degrees=10),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    ],
    p=0.7,
)
intensity_transforms = transforms.RandomApply(
    [transforms.ColorJitter(brightness=0.2, contrast=0.2)], p=0.7
)

train_transform = transforms.Compose(
    [
        transforms.Resize((128, 128)),
        transforms.Grayscale(num_output_channels=1),
        geometric_transforms,
        intensity_transforms,
        transforms.ToTensor(),
        transforms.Normalize(mean=[mean], std=[std]),
    ]
)

In [ ]:
# apply transforms
train_dataset.transform = train_transform
val_dataset.transform = preprocessing
test_dataset.transform = preprocessing
test_no_normalization_dataset.transform = preprocessing_no_normalization

# show train samples after preprocessing+augmenting
show_samples(train_dataset)

# loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=1)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=1)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=1)
test_no_normalization_loader = DataLoader(test_no_normalization_dataset, batch_size=64, shuffle=False, num_workers=1)


We export the preprocessed and augmented data to numpy arrays so they can be reused in other notebooks 
without reapplying the transformations. The training set is doubled by combining the original 
preprocessed images with an augmented version, effectively increasing the training set size.
The channel dimension is changed from `(N, C, H, W)` to `(N, H, W, C)` to match 
the expected input format for non-PyTorch models used in later tasks.

In [ ]:
"""Export the data to numpy arrays so we can use it in other notebooks without
needing to load the data again and apply the transformations again. """
from torch.utils.data import ConcatDataset

def loader_to_numpy(loader):
    X, y = [], []
    for images, labels in loader:
        X.append(images.numpy())
        y.append(labels.numpy())
    return np.concatenate(X), np.concatenate(y)


# create a copy of the train dataset with only preprocessing (no augmentation)
train_dataset_original = ImageFolder(root=base_path + "/train", transform=preprocessing)

# create a copy with augmentation
train_dataset_augmented = ImageFolder(root=base_path + "/train", transform=train_transform)

# combine both
train_dataset_combined = ConcatDataset([train_dataset_original, train_dataset_augmented])

# create loader from combined dataset
train_loader_combined = DataLoader(train_dataset_combined, batch_size=64, shuffle=True, num_workers=1)

# export to numpy
X_train, y_train = loader_to_numpy(train_loader_combined)
X_train = np.transpose(X_train, (0, 2, 3, 1))

X_val, y_val     = loader_to_numpy(val_loader)
X_test, y_test = loader_to_numpy(test_loader)
X_test_no_norm, y_test_no_norm = loader_to_numpy(test_no_normalization_loader)

X_val = np.transpose(X_val, (0, 2, 3, 1))
X_test = np.transpose(X_test, (0, 2, 3, 1))
X_test_no_norm = np.transpose(X_test_no_norm, (0, 2, 3, 1))

np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_val.npy",   X_val)
np.save("y_val.npy",   y_val)
np.save("X_test.npy",  X_test)
np.save("y_test.npy",  y_test)
np.save("X_test_no_norm.npy",  X_test_no_norm)
np.save("y_test_no_norm.npy",  y_test_no_norm)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# laad de data
X_test = np.load("X_test_no_norm.npy")
y_test = np.load("y_test_no_norm.npy")

# aantal afbeeldingen dat je wil tonen
num_images = 6

# kies willekeurige indices
indices = np.arange(len(X_test))[:num_images]

# plot de afbeeldingen
plt.figure(figsize=(10, 6))

for i, idx in enumerate(indices):
    plt.subplot(2, 3, i + 1)
    plt.imshow(X_test_no_norm[idx].squeeze(), cmap="gray")
    plt.title(f"Label: {y_test_no_norm[idx]}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# laad de data
X_test = np.load("X_test.npy")
y_test = np.load("y_test.npy")

# aantal afbeeldingen dat je wil tonen
num_images = 6

# kies willekeurige indices
indices = np.arange(len(X_test))[:num_images]

# plot de afbeeldingen
plt.figure(figsize=(10, 6))

for i, idx in enumerate(indices):
    plt.subplot(2, 3, i + 1)
    plt.imshow(X_test[idx].squeeze(), cmap="gray")
    plt.title(f"Label: {y_test[idx]}")
    plt.axis("off")

plt.tight_layout()
plt.show()